In [1]:
import os
import warnings
import time

import pandas as pd
import numpy as np
import math

import pickle

from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.graph_objects as go

import lifelines
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw

import tensorflow as tf
import tensorflow_probability as tfp

config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.compat.v1.Session(config = config)

from tensorflow import keras
from tensorflow.keras import optimizers, initializers, regularizers, layers

import scipy.stats as stats
from scipy.stats import norm, t, probplot, pearsonr, spearmanr, rankdata
from scipy.stats import truncnorm as truncnorm_scipy
from scipy.stats import gamma as gamma_dist
from scipy.special import gamma

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold

# import thetaflow as thf
import modelnn2 as thf

import json
import gc
import glob
from pathlib import Path

2026-05-30 05:06:23.529684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780128383.546609   54510 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780128383.551633   54510 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780128383.564537   54510 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780128383.564559   54510 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780128383.564561   54510 computation_placer.cc:177] computation placer alr

# Data Loading

In [2]:
import pyarrow.parquet as pq

# Parquet file path
parquet_file_path = "train_data.parquet"
parquet_reader = pq.ParquetFile(parquet_file_path)

def tabular_batch_generator(time_col, event_col, batch_size):
    time_col = time_col.decode('utf-8')
    event_col = event_col.decode('utf-8')
    
    # Pull data from parquet file iteratively in chunks
    for batch in parquet_reader.iter_batches(batch_size = batch_size):
        # Convert table to pandas DataFrame for column sclicing
        df = batch.to_pandas()

        # Obtain the time and censorship information from the dataset
        time = df[time_col].values.reshape(-1, 1)
        event = df[event_col].values.reshape(-1, 1)
        # Remove the response variable values from the table
        X = df.drop(columns = [time_col, event_col])
        
        # Yield the exact tuple structure thetaflow expects: (X, time, event) as a generator
        yield (X, time, event)

In [3]:
train_batch_size = 50000
train_ds = tf.data.Dataset.from_generator(
    tabular_batch_generator,
    args=("tempo", "delta", train_batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 100), dtype = tf.float32),
        tf.TensorSpec(shape=(None,1), dtype = tf.float32),
        tf.TensorSpec(shape=(None,1), dtype = tf.float32)
    )
)

num_batches = int(np.ceil( 11263238 / train_batch_size ))

# Keep the GPU fed during optimization
train_ds = (
    train_ds
    .apply( tf.data.experimental.assert_cardinality(num_batches) )
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
test_batch_size = 50000
test_ds = tf.data.Dataset.from_generator(
    tabular_batch_generator,
    args=("tempo", "delta", test_batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 100), dtype = tf.float32),
        tf.TensorSpec(shape=(None,1), dtype = tf.float32),
        tf.TensorSpec(shape=(None,1), dtype = tf.float32)
    )
)

num_batches_test = int(np.ceil( 11263238 / train_batch_size ))
# Keep the GPU fed during optimization
train_ds = (
    train_ds
    .apply( tf.data.experimental.assert_cardinality(num_batches_test) )
    .prefetch(tf.data.AUTOTUNE)
)

In [4]:
cardinality = tf.data.experimental.cardinality(train_ds).numpy()
is_cardinality_known = cardinality not in [tf.data.experimental.UNKNOWN_CARDINALITY, tf.data.experimental.INFINITE_CARDINALITY]
is_cardinality_known

True

In [92]:
batch_sizes = []
for batch_full_data in train_ds:
    X, times, events = batch_full_data
    batch_sizes.append( X.shape )

In [20]:
parquet_file = pq.ParquetFile('train_data.parquet')
table = parquet_file.read_row_group(0)
print(table.shape)
table.to_pandas().tail(3)

(21999, 102)


,tempo,delta,idade,horas_semanais,remuneracao_media_nominal,qtd_dias_afastamento,sexo_feminino,raca_amarela,raca_branca,raca_indigena,...,tamanho_estabelecimento_sem funcionarios,grupo_ocupacao_cbo_agropecuaria,grupo_ocupacao_cbo_diretores gerentes,grupo_ocupacao_cbo_industria producao,grupo_ocupacao_cbo_manutencao reparacao,grupo_ocupacao_cbo_profissionais nivel superior,grupo_ocupacao_cbo_servicos administrativos,grupo_ocupacao_cbo_tecnicos nivel medio,regime_jornada_jornada intermitente,regime_jornada_jornada parcial
21996,8.400000,1,-1.193314,0.399377,-0.377034,-0.159835,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
21997,113.300003,0,0.626800,-0.632820,-0.300957,-0.159835,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21998,10.400000,1,-0.846626,0.399377,-0.449153,-0.159835,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [88]:
batch_generator = parquet_file.iter_batches(batch_size = 21996)
batch = next(batch_generator).to_pandas()
batch.tail(5)

,tempo,delta,idade,horas_semanais,remuneracao_media_nominal,qtd_dias_afastamento,sexo_feminino,raca_amarela,raca_branca,raca_indigena,...,tamanho_estabelecimento_sem funcionarios,grupo_ocupacao_cbo_agropecuaria,grupo_ocupacao_cbo_diretores gerentes,grupo_ocupacao_cbo_industria producao,grupo_ocupacao_cbo_manutencao reparacao,grupo_ocupacao_cbo_profissionais nivel superior,grupo_ocupacao_cbo_servicos administrativos,grupo_ocupacao_cbo_tecnicos nivel medio,regime_jornada_jornada intermitente,regime_jornada_jornada parcial
21991,100.300003,0,0.020095,0.399377,-0.124406,-0.159835,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
21992,3.100000,1,-0.673282,0.399377,-0.435828,-0.159835,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
21993,6.700000,1,-0.413265,0.399377,-0.423830,-0.159835,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21994,40.400002,1,-0.673282,0.399377,0.228513,-0.159835,1,0,1,0,...,0,0,0,0,0,0,1,0,0,0
21995,14.200000,1,0.540128,0.399377,-0.308778,-0.069544,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [90]:
type(next(batch_generator))

pyarrow.lib.RecordBatch

In [73]:
11263238 / 50000

225.26476

In [76]:
batch_generator = parquet_file.iter_batches(batch_size = 50000)
batch_shapes = []
for i in range(226):
    batch = next(batch_generator).to_pandas()
    batch_shapes.append( batch.shape )

In [57]:
parquet_file = pq.ParquetFile('train_data.parquet')
table = parquet_file.read_row_group(0)
print(table.shape)
table.to_pandas().tail(5)

(21999, 102)


,tempo,delta,idade,horas_semanais,remuneracao_media_nominal,qtd_dias_afastamento,sexo_feminino,raca_amarela,raca_branca,raca_indigena,...,tamanho_estabelecimento_sem funcionarios,grupo_ocupacao_cbo_agropecuaria,grupo_ocupacao_cbo_diretores gerentes,grupo_ocupacao_cbo_industria producao,grupo_ocupacao_cbo_manutencao reparacao,grupo_ocupacao_cbo_profissionais nivel superior,grupo_ocupacao_cbo_servicos administrativos,grupo_ocupacao_cbo_tecnicos nivel medio,regime_jornada_jornada intermitente,regime_jornada_jornada parcial
21994,40.400002,1,-0.673282,0.399377,0.228513,-0.159835,1,0,1,0,...,0,0,0,0,0,0,1,0,0,0
21995,14.200000,1,0.540128,0.399377,-0.308778,-0.069544,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21996,8.400000,1,-1.193314,0.399377,-0.377034,-0.159835,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
21997,113.300003,0,0.626800,-0.632820,-0.300957,-0.159835,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21998,10.400000,1,-0.846626,0.399377,-0.449153,-0.159835,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
dataset_iterator = iter(train_ds)
X_batch, y_batch, delta_batch = next(dataset_iterator)

In [15]:
X_batch

<tf.Tensor: shape=(256, 100), dtype=float32, numpy=
array([[ 0.80014443,  0.39937726, -0.3844489 , ...,  0.        ,
         0.        ,  0.        ],
       [-1.01997   ,  0.39937726, -0.40069282, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.71347237,  0.39937726, -0.2719538 , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.15324883,  0.39937726,  0.8135426 , ...,  0.        ,
         0.        ,  0.        ],
       [-0.4999373 , -0.63281983, -0.33557734, ...,  0.        ,
         0.        ,  0.        ],
       [-0.7599537 ,  0.39937726,  0.0498611 , ...,  0.        ,
         0.        ,  0.        ]], dtype=float32)>

# Models

In [5]:
def build_weibull_model( ridge_penalty = 0.01, lasso_penalty = 0.01 ):
    # Shape parameter: constant for all patients
    # Scale parameter: modeled as a neural network output
    weibull_parameters = {
        "k": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "independent", "shape": 1, "init": 1.0, "warmup_time": 0},
        "lam": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "nn", "shape": 1, "init": 1.0, "warmup_time": 0}
    }

    def loglikelihood_loss(model, nn_output, data):
        # Unpack your data tuple
        X, y, delta = data
        
        k = model.get_variable("k")
        # k = model.get_variable("k", nn_output)
        log_lam = model.get_variable("lam", nn_output, get_raw_value = True)

        log_y = tf.math.log(y + 1.0e-7)
        log_hazard = tf.math.log(k) - k * log_lam + (k - 1.0) * log_y
        survival_term = tf.math.exp(k * (log_y - log_lam))

        loglik_terms = (delta * log_hazard) - survival_term
        neg_loglik = -tf.reduce_sum(loglik_terms)
        
        return neg_loglik

    def neural_network(model, seed = None):
        initializer = initializers.GlorotNormal(seed = seed)
        elastic_net = tf.keras.regularizers.L1L2(l1 = lasso_penalty, l2 = ridge_penalty)
        model.dense1 = layers.Dense(
            units = 32,
            activation = "gelu",
            kernel_initializer = initializer,
            kernel_regularizer = elastic_net,
            use_bias = True,
            dtype = tf.float32, 
            name = "gene_bottleneck"
        )
        model.dense2 = layers.Dense(
            units = 16,
            activation = "gelu",
            kernel_initializer = initializer,
            use_bias = True,
            dtype = tf.float32,
            name = "hidden_layer1"
        )
        model.dense3 = layers.Dense(
            units = 2,
            activation = "gelu",
            kernel_initializer = initializer,
            use_bias = True,
            dtype = tf.float32,
            name = "statistical_layer"
        )
        model.output_layer = layers.Dense(
            units = 1,
            activation = None,
            use_bias = True,
            kernel_initializer = initializer,
            dtype = tf.float32,
            name = "log_lambda"
        )
    
    def neural_network_call(model, x_input, training = False):
        x = model.dense1(x_input)
        x = model.dense2(x)
        x = model.dense3(x)
        x = model.output_layer(x)
        return x
    
    def neural_network_call_nolast(model, x_input):
        x = model.dense1(x_input)
        x = model.dense2(x)
        x = model.dense3(x)
        return x

    return weibull_parameters, loglikelihood_loss, neural_network, neural_network_call, neural_network_call_nolast

In [6]:
with tf.device("/GPU:0"):
    weibull_parameters, weibull_loss, weibull_neural_network, weibull_call, weibull_call_nolast = \
    build_weibull_model( ridge_penalty = 0.01, lasso_penalty = 0.01 )
    seed = 10
    weibull_model = thf.ModelNN(weibull_parameters, weibull_loss,
                                weibull_neural_network, weibull_call,
                                weibull_call_nolast, input_dim = (100,), seed = seed)
    weibull_model.pre_train_model(epochs = None, x = train_ds, data = None, n_train = 11263238, shuffle = True)
    weibull_model.train_model(epochs = 5000, x = train_ds, data = None, n_train = 11263238,
                              shuffle = True,
                              get_covariances = True,
                              validation = False, val_prop = 0.2, force_training_validation = False,
                              optimizer_independent = optimizers.Adam(learning_rate = 0.01, clipnorm = 1.0),
                              optimizer_nn = optimizers.Adam(learning_rate = 0.01, clipnorm = 1.0),
                              fine_tune_nn_lr = 0.01, fine_tune_independent_lr = 0.01,
                              early_stopping = True, early_stopping_patience = 50, 
                              early_stopping_warmup = 10,
                              reduce_lr = True, reduce_lr_warmup = 0,
                              reduce_lr_factor = 0.5, reduce_lr_min_delta = 1.0e-3, reduce_lr_patience = 25,
                              reduce_lr_cooldown = 10, reduce_lr_min_lr = 1.0e-5,
                              fine_tune = True,
                              finetune_early_stopping = True, finetune_early_stopping_patience = 50,
                              finetune_early_stopping_warmup = 10,
                              finetune_reduce_lr = True, finetune_reduce_lr_warmup = 0,
                              finetune_reduce_lr_factor = 0.5, finetune_reduce_lr_min_delta = 1.0e-2, finetune_reduce_lr_patience = 25,
                              finetune_reduce_lr_cooldown = 10, finetune_reduce_lr_min_lr = 1.0e-5,
                              deterministic = True,
                              verbose = True, print_freq = 1,
                              train_batch_size = None, val_batch_size = None,
                              buffer_size = 4096, gradient_accumulation_steps = None,)

ValueError: When providing a batched tf.data.Dataset, you must explicitly provide `n_train` (the total number of training samples).

In [4]:
def converter_parquet_para_tfrecords(parquet_path, output_dir, num_shards=32):
    """
    Lê o arquivo Parquet pelos Row Groups nativos e converte em shards de TFRecord.
    """
    os.makedirs(output_dir, exist_ok=True)
    parquet_file = pq.ParquetFile(parquet_path)
    num_groups = parquet_file.num_row_groups
    
    # Nomes das colunas conforme seu dicionário atual
    time_col = "tempo"
    event_col = "delta"
    
    # Inicializa os writers binários para os shards (escrita balanceada)
    writers = [
        tf.io.TFRecordWriter(os.path.join(output_dir, f"shard_{i:03d}.tfrecord"))
        for i in range(num_shards)
    ]
    
    print(f"Iniciando conversão de {num_groups} Row Groups para {num_shards} TFRecords...")
    
    for i in range(num_groups):
        table = parquet_file.read_row_group(i)
        df_batch = table.to_pandas()
        
        # Separação dos alvos de sobrevivência
        times = df_batch[time_col].values.astype(np.float32)
        events = df_batch[event_col].values.astype(np.float32)
        
        # Separação e isolamento das exatas 100 features restantes
        df_features = df_batch.drop(columns=[time_col, event_col])
        features_matrix = df_features.values.astype(np.float32)
        
        # Iteração por linha dentro do Row Group para criar os buffers de protocolo
        for j in range(len(df_batch)):
            # Converte vetores e escalares em tipos reconhecidos pelo Protobuf do TF
            feature_list = tf.train.Feature(float_list=tf.train.FloatList(value=features_matrix[j]))
            time_feature = tf.train.Feature(float_list=tf.train.FloatList(value=[times[j]]))
            event_feature = tf.train.Feature(float_list=tf.train.FloatList(value=[events[j]]))
            
            # Monta o dicionário estruturado do registro
            example = tf.train.Example(features=tf.train.Features(feature={
                'features': feature_list,
                'time': time_feature,
                'event': event_feature
            }))
            
            # Distribui as linhas entre os shards via técnica round-robin (j % num_shards)
            writers[j % num_shards].write(example.SerializeToString())
            
        if (i + 1) % 50 == 0 or i == num_groups - 1:
            print(f"Progresso: {i + 1}/{num_groups} Row Groups serializados com sucesso.")
            
    # Fecha todos os arquivos abertos de escrita
    for w in writers:
        w.close()
    print("Conversão de dados binários concluída!")

# Execução única do pipeline de dados
converter_parquet_para_tfrecords("train_data.parquet", "tfrecords_dir_train", num_shards = 32)

Iniciando conversão de 512 Row Groups para 32 TFRecords...
Progresso: 50/512 Row Groups serializados com sucesso.
Progresso: 100/512 Row Groups serializados com sucesso.
Progresso: 150/512 Row Groups serializados com sucesso.
Progresso: 200/512 Row Groups serializados com sucesso.
Progresso: 250/512 Row Groups serializados com sucesso.
Progresso: 300/512 Row Groups serializados com sucesso.
Progresso: 350/512 Row Groups serializados com sucesso.
Progresso: 400/512 Row Groups serializados com sucesso.
Progresso: 450/512 Row Groups serializados com sucesso.
Progresso: 500/512 Row Groups serializados com sucesso.
Progresso: 512/512 Row Groups serializados com sucesso.
Conversão de dados binários concluída!


In [6]:
converter_parquet_para_tfrecords("test_data.parquet", "tfrecords_dir_test", num_shards = 16)

Iniciando conversão de 256 Row Groups para 16 TFRecords...
Progresso: 50/256 Row Groups serializados com sucesso.
Progresso: 100/256 Row Groups serializados com sucesso.
Progresso: 150/256 Row Groups serializados com sucesso.
Progresso: 200/256 Row Groups serializados com sucesso.
Progresso: 250/256 Row Groups serializados com sucesso.
Progresso: 256/256 Row Groups serializados com sucesso.
Conversão de dados binários concluída!
